In [1]:
import pandas as pd

In [2]:
model_df = pd.read_parquet("../data/processed/lastfm_scrobbles_with_tags.parquet")

In [54]:
model_df.memory_usage(deep=True).sum() / 1e9

np.float64(0.034349443)

In [ ]:
model_df["tags_clean"][0]

In [11]:
model_df["artist_clean"].isna().mean()

np.float64(0.0012908009832058552)

In [20]:
model_df["track_clean"].isna().mean()

np.float64(0.0)

In [3]:
model_df["artist_clean"] = model_df["artist_clean"].fillna("")

In [4]:
def remove_artist_tag(tags, artist_name):
    artist_name = artist_name.lower()
    return [t for t in tags if artist_name not in t]

In [5]:
def clean_tag(tag):
    tag = tag.lower().strip()
    tag = tag.replace("-", " ")
    return tag

In [6]:
TAG_MAP = {
    "alternative rock": "rock",
    "indie rock": "indie",
    "indie pop": "indie pop",
    "electronica": "electronic",
    "r&b": "rnb",
    "rhythm and blues": "rnb",
    "synth pop": "synthpop",
    "lo fi": "lo fi",
    "hip hop": "hip hop",
    "pop rock": "rock",
}

In [7]:
def normalize_tags(tags, artist_name):
    normalized = []
    
    for tag in tags:
        
        if artist_name in tag:
            continue
        
        tag = clean_tag(tag)
        tag = TAG_MAP.get(tag, tag)
        normalized.append(tag)
    
    return list(dict.fromkeys(normalized))  # remove duplicates while preserving order

In [8]:
REMOVE_METADATA_TAGS = {
    "british", "scottish", "american", "polish", "irish", "usa", "canadian", "english", "french", "german", "swedish", "japanese", "welsh", "manchester",
    "20s", "30s", "40s", "50s", "60s", "70s", "80s", "90s", "00s", "10s",
    "female vocalists", "male vocalists", "singer songwriter",
    "rock", "pop", "electronic"
}

def filter_metadata_tags(tags):
    return [tag for tag in tags if tag not in REMOVE_METADATA_TAGS]

In [9]:
model_df["tags_normalized"] = model_df.apply(lambda row: normalize_tags(row["tags_clean"], row["artist_clean"]), axis=1)
model_df["tags_filtered"] = model_df["tags_normalized"].apply(filter_metadata_tags)

In [ ]:
# below filtering was important for MultiLabelBinarizer (count-based). In TF-IDF common tags are downweighted, and rare tags are upweighted, so it was less important to remove rare tags.

In [ ]:
# from collections import Counter

# all_tags = [t for tags in model_df["tags_filtered"] for t in tags]
# tag_counts = Counter(all_tags)

# MIN_COUNT = 5

# def remove_rare_tags(tags):
#     return [t for t in tags if tag_counts[t] >= MIN_COUNT]

In [ ]:
#model_df["tags_final"] = model_df["tags_filtered"].apply(remove_rare_tags)

In [10]:
model_df["tags_filtered"][0]

['indie', 'alternative', 'britpop']

In [12]:
model_df.to_parquet("../data/processed/lastfm_scrobbles_clean_tags_final_tfid.parquet", index=False)
model_df.to_csv("../data/processed/lastfm_scrobbles_clean_tags_final_tfid.csv", index=False)